# 07 RAG & Agents：检索增强与智能体

> 前置：`01-linear-algebra/02`（向量与内积）、`01-tokenization`。
> 目标：理解 RAG 为什么能"让模型读到资料库"；实现一个可运行的向量检索演示；拆解 Function Calling / Agent 循环。

## RAG（Retrieval-Augmented Generation）

LLM 的硬伤：**知识截止于训练数据、会幻觉、无法访问私有库**。RAG 的解法是"先检索、再回答"：

```
query → 向量化 → 在知识库中余弦相似度检索 top-k
                    ↓
    上下文 + query → LLM → 回答（有据可依）
```

优点：知识可更新（换库即可）、可溯源（附来源）、成本低（不用重新训练）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 向量检索演示

用"字符袋 + 词频加权"（TF 思想）把句子变成向量，余弦相似度检索。真实的 RAG 用 embedding 模型（如 text-embedding-3、bge），但**检索机制完全一样**。

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "the bird sang in the tree",
    "the fox jumped over the fence",
    "the sun rose in the east",
    "the rain fell on the roof",
    "the cat chased the mouse",
    "a dog barked at the mailman",
]
vocab = sorted({c for s in corpus for c in s if c != " "})
print("字符词表大小:", len(vocab))

def bow_vec(s):
    v = np.zeros(len(vocab))
    for i, c in enumerate(vocab):
        v[i] = s.count(c)                      # 字符频次（词频的简化版）
    return v / (np.linalg.norm(v) + 1e-8)      # 归一化 → 余弦等价于内积

docs = np.array([bow_vec(s) for s in corpus])

def retrieve(query, k=2):
    qv = bow_vec(query)
    scores = docs @ qv
    top = np.argsort(-scores)[:k]
    return [(corpus[i], scores[i]) for i in top]

print("\n查询 'dog barks':")
for s, sc in retrieve("dog barks", k=2):
    print(f"  {sc:.3f}  {s}")
print("\n查询 'sun in the sky':")
for s, sc in retrieve("sun in the sky", k=2):
    print(f"  {sc:.3f}  {s}")

## RAG 提示词组装

检索到的文档拼进 prompt 模板——模型"带着材料回答"。用上一课的 TinyGPT 展示"检索 → 提示 → 生成"的完整链路。

In [ ]:
def rag_prompt(query, k=2):
    hits = retrieve(query, k=k)
    ctx = "\n".join(f"- {s}" for s, _ in hits)
    return f"根据以下资料回答问题。\n\n资料：\n{ctx}\n\n问题：{query}\n回答："

print(rag_prompt("what does the cat do?", k=2))

## Function Calling 与 Agent 循环

**Function Calling**：LLM 不直接执行工具，而是**输出结构化的调用意图**（JSON），由系统执行并回填结果。这是 OpenAI/Anthropic 的标配能力。

**Agent 循环**（ReAct 思想）：

```
循环:
  LLM 根据 用户问题 + 历史 + 工具列表 → 决定: 回答问题 或 调用工具(JSON)
  若调用工具 → 执行 → 把结果追加到上下文 → 回到循环
  直到 LLM 直接回答
```

In [ ]:
import json

tools = {
    "get_weather": {"desc": "查询城市天气", "params": {"city": "string"}},
    "calculate": {"desc": "数学计算", "params": {"expr": "string"}},
}

def call_tool(name, args):
    if name == "get_weather":
        return f"{{'city': '{args['city']}', 'weather': '晴 26°C'}}"
    if name == "calculate":
        return f"{{'expr': '{args['expr']}', 'result': {eval(args['expr'])}}}"
    return "unknown tool"

# 模拟：模型输出 Function Call（真实场景由 LLM 生成这段 JSON）
model_call = json.dumps({"tool": "calculate", "args": {"expr": "6 * 7"}})
print("模型输出:", model_call)
call = json.loads(model_call)
result = call_tool(call["tool"], call["args"])
print("执行结果:", result)

# Agent 循环骨架
messages = [
    {"role": "user", "content": "北京天气怎么样？"},
    {"role": "assistant", "content": json.dumps({"tool": "get_weather", "args": {"city": "北京"}})},
    {"role": "tool", "content": call_tool("get_weather", {"city": "北京"})},
    {"role": "assistant", "content": "北京今天晴，26°C。"},
]
print("\nAgent 多轮消息流（简版）:")
for m in messages:
    print(f"  [{m['role']}] {m['content'][:40]}")

## 总结

1. **RAG 解决"知识"问题**：向量化 + 余弦检索 + 提示组装——三行代码就是最小实现，生产级再加 embedding 模型、向量数据库（Milvus/pgvector）、rerank。
2. **Function Calling 解决"行动"问题**：LLM 输出 JSON 意图 → 系统执行 → 结果回填，让模型"能做事"。
3. **Agent = 循环**：检索（RAG）+ 工具（FC）+ 记忆（上下文历史）+ 规划（ReAct 推理）拼成自主智能体。
4. **局限**：检索质量决定上限（垃圾进垃圾出）；Agent 循环有累积错误与成本风险，需护栏。

## 课后练习

1. 把 `bow_vec` 的字符频次换成"去掉停用词后的词频"，对比检索质量——为什么"the/on"会污染相似度？
2. 给 retrieve 加一个 "rerank"：先用字符袋粗筛 top-5，再用更精细的向量（如 bigram 频次）精排 top-2。
3. 用 TinyGPT 做一步"工具选择"：给定工具描述和用户问题，让模型输出工具名（监督训练一个分类头）。